# Eigenvalues and Eigenvectors

Companion notebook for the [Eigenvalues and Eigenvectors](https://ml-viz-ruby.vercel.app/courses/linear-algebra/03-eigenvalues-and-eigenvectors) lesson.

We'll compute eigendecompositions, visualize eigenvectors geometrically, and implement PCA from scratch.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Intuition — the directions a matrix leaves alone

Most vectors get both rotated and stretched when you multiply them by a matrix. But
almost every matrix has a few special directions — its **eigenvectors** — that it only
*stretches*, never turns: `A v = λ v`, where the scalar `λ` (the **eigenvalue**) is
the stretch factor. Those directions are the matrix's "natural axes." Find them and
you understand what the matrix really does — which is why eigenvectors sit under PCA
(directions of maximum variance), PageRank (the dominant eigenvector of the web
graph), stability analysis, and the diagonalization that makes repeated application
cheap.

## 1. Eigenvalues from scratch — the characteristic polynomial

`np.linalg.eig` is a black box; here's what it's really doing. An eigenvalue makes
`A − λI` **singular** (it must squash the eigenvector to zero), so it solves
`det(A − λI) = 0`. For a 2×2 matrix that determinant expands to a neat quadratic,

$$\lambda^2 - \operatorname{tr}(A)\,\lambda + \det(A) = 0,$$

whose two roots are the eigenvalues. Each eigenvector is then a null-space direction of
`(A − λI)`. As a built-in sanity check, the eigenvalues must **sum to the trace** and
**multiply to the determinant**.

In [ ]:
A = np.array([[4., 1.], [2., 3.]])

# Derive eigenvalues BY HAND from the characteristic equation det(A - λI) = 0.
# For 2x2 this is  λ² - tr(A)·λ + det(A) = 0.
tr  = np.trace(A)
det = np.linalg.det(A)
print(f'characteristic poly:  λ² - {tr:.0f}λ + {det:.0f} = 0')

disc = tr**2 - 4 * det
lam1 = (tr + np.sqrt(disc)) / 2
lam2 = (tr - np.sqrt(disc)) / 2
print(f'roots (eigenvalues):  λ1={lam1:.0f}, λ2={lam2:.0f}')

# Eigenvector = null space direction of (A - λI). For a 2x2 singular matrix
# [[a,b],[c,d]] a null vector is [b, -a] (or [-d, c]); pick whichever is non-zero.
def eigvec(A, lam):
    M = A - lam * np.eye(2)
    v = np.array([M[0, 1], -M[0, 0]])
    if np.allclose(v, 0):
        v = np.array([-M[1, 1], M[1, 0]])
    return v / np.linalg.norm(v)

for lam in (lam1, lam2):
    v = eigvec(A, lam)
    print(f'λ={lam:.0f}: eigenvector {v.round(3)}  check A·v - λ·v = {(A @ v - lam * v).round(6)}')

# Eigenvalues tie back to trace and determinant:
print(f'\nsum of eigenvalues  = {lam1 + lam2:.0f}  = trace(A) = {tr:.0f}')
print(f'product of eigenvalues = {lam1 * lam2:.0f}  = det(A)  = {det:.0f}')

**What to notice:** the hand-derived roots are **5** and **2**, the eigenvectors
satisfy `A·v − λ·v ≈ 0`, and the invariants hold: `5 + 2 = 7 = trace` and
`5 × 2 = 10 = det`. No linear-algebra library was called — just the quadratic
formula and the definition.

## 2. The library way — `np.linalg.eig`, cross-checked

In practice you call `np.linalg.eig`, which returns the eigenvalues in an array and the
eigenvectors as the **columns** of a matrix (so `eigenvectors[:, i]` pairs with
`eigenvalues[i]` — a common indexing trap). The cell below runs it, **asserts its
eigenvalues match the hand-derived 5 and 2**, and verifies the defining equation
`A v = λ v` for each pair.

In [ ]:
A = np.array([[4., 1.], [2., 3.]])

eigenvalues, eigenvectors = np.linalg.eig(A)
print('Matrix A:')
print(A)
print('\nEigenvalues:', eigenvalues)
print('Eigenvectors (columns):')
print(eigenvectors)

# Verify: A @ v == λ * v
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]
    av  = A @ v
    lv  = lam * v
    print(f'\nλ={lam:.1f}: Av={av.round(4)}, λv={lv.round(4)}, equal={np.allclose(av,lv)}')

# cross-check against the from-scratch derivation above
assert np.allclose(sorted(eigenvalues), sorted([lam1, lam2])), "must match by-hand roots"
print('\nnp.linalg.eig matches the by-hand eigenvalues ✓')

**What to notice:** the eigenvalues come out as **5** and **2** (matching §1), and each
`Av` lands exactly on `λv` (`equal=True`). Along an eigenvector, the matrix behaves like
plain scalar multiplication — that's the entire idea, now confirmed two independent ways.

## 3. Seeing eigenvectors — special vs ordinary directions

For a random vector, `A` changes both direction and length. For an eigenvector, only
the length changes. The figure draws each eigenvector (solid) and its transform `A·v`
(dashed) alongside a random vector and *its* transform, so the difference is visible.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

colors = ['#6366f1', '#2dd4bf', '#f97316']

# Plot eigenvectors and their transforms
for i, color in zip(range(2), colors[:2]):
    v   = eigenvectors[:, i]
    Av  = A @ v
    lam = eigenvalues[i]

    ax.annotate('', xy=v, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    ax.annotate('', xy=Av, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color=color, lw=2, linestyle='--', alpha=0.6))
    ax.text(v[0]*1.05, v[1]*1.05 + 0.1, f'v₁  (λ={lam:.0f})', color=color, fontsize=11)

# Random vector
r  = np.array([1.0, 0.3])
Ar = A @ r
ax.annotate('', xy=r, xytext=[0,0], arrowprops=dict(arrowstyle='->', color=colors[2], lw=2))
ax.annotate('', xy=Ar, xytext=[0,0],
            arrowprops=dict(arrowstyle='->', color=colors[2], lw=2, linestyle='--', alpha=0.6))
ax.text(r[0]*1.1, r[1]+0.1, 'random →', color=colors[2], fontsize=10)
ax.text(Ar[0]*1.05, Ar[1]+0.1, '→ A·random', color=colors[2], alpha=0.7, fontsize=10)

ax.set_xlim(-0.3, 2); ax.set_ylim(-0.3, 2.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
ax.axhline(0, color='#30344a'); ax.axvline(0, color='#30344a')
ax.set_title('Eigenvectors stay in the same direction after transformation', pad=12)
plt.tight_layout(); plt.show()

**What to notice:** each eigenvector and its transform lie on the **same line** — the
matrix only lengthened it by `λ`. The orange **random** vector, by contrast, visibly
swings to a new direction under `A`. Eigenvectors are exactly the directions where that
swing is zero.

## PCA from scratch — eigenvectors of the covariance

Principal Component Analysis is eigendecomposition applied to the **covariance matrix**:
its eigenvectors are the axes the data varies along, and each eigenvalue is the
**variance** captured on that axis. The cell builds correlated 2-D data, forms the
covariance `C = XᵀX / n`, and uses `np.linalg.eigh` (the symmetric solver — see the
gotchas below), sorting eigenvalues **descending** so the top one is **PC1**. Projecting
onto PC1, `X @ V[:, 0]`, compresses 2-D to 1-D.

In [ ]:
rng = np.random.default_rng(42)

# Correlated 2D data
cov_true = np.array([[3., 2.], [2., 2.]])
X = rng.multivariate_normal([0, 0], cov_true, size=200)

# PCA via eigendecomposition
C = (X.T @ X) / len(X)                    # sample covariance
eigenvalues, V = np.linalg.eigh(C)          # eigh for symmetric
idx = np.argsort(eigenvalues)[::-1]         # sort descending
eigenvalues, V = eigenvalues[idx], V[:, idx]

explained = eigenvalues / eigenvalues.sum()
print('Eigenvalues:', eigenvalues.round(4))
print('Variance explained:', explained.round(4))
print('PC1 direction:', V[:, 0].round(4))

# Project to 1D
X_1d = X @ V[:, 0]       # principal component scores

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], alpha=0.4, s=15, color='#6366f1')
scale = 2
for i, color in enumerate(['#f97316', '#2dd4bf']):
    v = V[:, i] * np.sqrt(eigenvalues[i]) * scale
    ax.annotate('', xy=v, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(v[0]+0.1, v[1]+0.1, f'PC{i+1} ({explained[i]*100:.0f}%)', color=color, fontsize=11)
ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
ax.set_title('Original data + Principal Components')

ax2 = axes[1]
ax2.hist(X_1d, bins=30, color='#6366f1', alpha=0.8, edgecolor='#0f1117')
ax2.set_title(f'Projected onto PC1 ({explained[0]*100:.0f}% variance)')
ax2.set_xlabel('PC1 score'); ax2.set_ylabel('Count')

plt.tight_layout(); plt.show()

**What to notice:** PC1 (orange) points straight down the cloud's long axis and alone
explains the large majority of the variance (~92%), with PC2 short and perpendicular.
The 1-D projection keeps most of the spread — which is exactly why PCA works: keep the
high-variance eigen-directions, drop the rest.

## 4. Limitations & numerical gotchas

- **Eigenvalues can be complex.** A pure rotation has *no* real invariant direction, so
  its eigenvalues are complex (`±i` for a 90° turn). `np.linalg.eig` returns complex
  dtype — don't assume real.
- **Symmetric matrices are special.** Covariance, Gram, and Hessian matrices are
  symmetric: their eigenvalues are **real** and eigenvectors **orthogonal**. Use
  `np.linalg.eigh` — it's faster and numerically stabler than `eig` and never returns
  spurious imaginary parts.
- **Defective matrices exist.** `[[1,1],[0,1]]` has the repeated eigenvalue 1 but only
  **one** independent eigenvector, so it *cannot* be diagonalized.
- **Output conventions bite.** `eig` eigenvalues are **unsorted**; eigenvectors are
  **columns** (not rows) and each has an arbitrary **sign**.

In [ ]:
# Complex eigenvalues: a 90° rotation has no real invariant direction
Rot = np.array([[0., -1.], [1., 0.]])
print('rotation eigenvalues:', np.linalg.eig(Rot)[0])          # ±1j

# Symmetric -> eigh gives real eigenvalues and orthogonal eigenvectors
Sym = np.array([[2., 1.], [1., 2.]])
vals, vecs = np.linalg.eigh(Sym)
print('eigh eigenvalues    :', vals.round(3), ' (real, ascending)')
print('eigenvectors orthogonal?', np.allclose(vecs.T @ vecs, np.eye(2)))

# Defective: repeated eigenvalue, rank-deficient eigenvector set
Def = np.array([[1., 1.], [0., 1.]])
w, V = np.linalg.eig(Def)
print('defective eigenvalues:', w.round(3), ' independent eigenvectors:',
      np.linalg.matrix_rank(V))

**What to notice:** the rotation's eigenvalues are `±1j` (complex); the symmetric
matrix gives clean real eigenvalues with orthogonal eigenvectors from `eigh`; and the
defective matrix has rank **1** worth of eigenvectors despite being 2×2 — proof it can't
be diagonalized. Knowing which case you're in tells you whether to call `eig`, `eigh`,
or reach for the SVD instead.

## Key takeaways

- An **eigenvector** is a direction `A` only scales: `A v = λ v`; the **eigenvalue** `λ`
  is the scale factor.
- For 2×2, eigenvalues solve `λ² − tr·λ + det = 0`; in general they **sum to the trace**
  and **multiply to the determinant**.
- `np.linalg.eig` returns eigenvectors as **columns**; use `np.linalg.eigh` for
  symmetric matrices (real eigenvalues, orthogonal eigenvectors, faster).
- **PCA is eigendecomposition of the covariance** — eigenvectors are principal
  directions, eigenvalues are variances.
- Watch for **complex** eigenvalues (rotations) and **defective** matrices (no full
  eigenbasis).

**Next:** [SVD & Low-Rank Approximation](https://ml-viz-ruby.vercel.app/courses/linear-algebra/04-svd-and-low-rank)
— eigen-style structure for *any* matrix, even non-square ones.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Eigenvalues from trace and determinant

For a 2×2 matrix, the characteristic polynomial collapses to a quadratic in two summaries you already know:

$$\lambda^2 - \text{tr}(A)\,\lambda + \det(A) = 0
\quad\Rightarrow\quad
\lambda = \frac{\text{tr}(A) \pm \sqrt{\text{tr}(A)^2 - 4\det(A)}}{2}$$

Implement it (assume real eigenvalues) and match `np.linalg.eigvals`.

In [ ]:
def eigvals_2x2(A):
    """Both eigenvalues of a 2x2 matrix (assumed real), returned (larger, smaller)."""
    A = np.asarray(A, dtype=float)

    # TODO(you): trace = sum of the diagonal
    tr = ...

    # TODO(you): determinant ad - bc
    det = ...

    # TODO(you): square root of the discriminant tr^2 - 4*det
    disc = ...

    return ((tr + disc) / 2, (tr - disc) / 2)

In [ ]:
# Checks — run me
big, small = eigvals_2x2([[4, 1], [2, 3]])
assert abs(big - 5) < 1e-12 and abs(small - 2) < 1e-12, "expected eigenvalues 5 and 2"

big, small = eigvals_2x2([[3, 0], [0, -1]])
assert abs(big - 3) < 1e-12 and abs(small + 1) < 1e-12, "diagonal matrix: eigenvalues sit on the diagonal"

A = np.array([[2.0, 1.0], [1.0, 2.0]])
assert np.allclose(sorted(eigvals_2x2(A)), sorted(np.linalg.eigvals(A).real)), "must match np.linalg.eigvals"

# Repeated eigenvalue (discriminant = 0): a scaled identity has a single eigenvalue twice
big, small = eigvals_2x2([[5, 0], [0, 5]])
assert abs(big - 5) < 1e-12 and abs(small - 5) < 1e-12, "a scaled identity has a repeated eigenvalue"

# A matrix with a zero eigenvalue (singular, rank-deficient)
big, small = eigvals_2x2([[1, 2], [2, 4]])
assert abs(small) < 1e-9, "a singular matrix has 0 as one of its eigenvalues"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def eigvals_2x2(A):
    A = np.asarray(A, dtype=float)
    tr = A[0, 0] + A[1, 1]
    det = A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]
    disc = np.sqrt(tr ** 2 - 4 * det)
    return ((tr + disc) / 2, (tr - disc) / 2)
```

</details>

### Exercise 2 — Power iteration

Repeatedly applying $A$ to *any* vector rotates it toward the **dominant eigenvector** — the component along the largest eigenvalue grows fastest, so after renormalizing each step, it's all that survives. The eigenvalue then falls out of the Rayleigh quotient $\lambda = \mathbf{v}^\top A \mathbf{v}$ (for unit $\mathbf{v}$). This is the idea behind PageRank and many sparse eigensolvers.

In [ ]:
def power_iteration(A, iters=100):
    """Return (dominant eigenvalue, unit eigenvector) by repeated multiplication."""
    A = np.asarray(A, dtype=float)
    v = np.ones(A.shape[0])

    for _ in range(iters):
        # TODO(you): multiply v by A
        v = ...
        # TODO(you): renormalize v to unit length
        v = ...

    # TODO(you): Rayleigh quotient v·(A v)  (v already has unit length)
    lam = ...

    return lam, v

In [ ]:
# Checks — run me
A = np.array([[3.0, 2.0], [2.0, 2.0]])
lam, v = power_iteration(A)

true_lams, true_V = np.linalg.eigh(A)
assert abs(lam - true_lams[-1]) < 1e-8, "should find the LARGEST eigenvalue"
assert np.linalg.norm(A @ v - lam * v) < 1e-8, "v must satisfy A v = λ v"
assert abs(abs(v @ true_V[:, -1]) - 1) < 1e-8, "v must align with the dominant eigenvector"

# Starting exactly ON an eigenvector should stay there (up to sign)
B = np.array([[4.0, 0.0], [0.0, -2.0]])
lam_b, v_b = power_iteration(B, iters=50)
assert abs(lam_b - 4) < 1e-8, "the dominant eigenvalue can be negative-adjacent but must be the largest in magnitude here"
assert abs(abs(v_b[0]) - 1) < 1e-8 and abs(v_b[1]) < 1e-8, "dominant eigenvector of a diagonal matrix is an axis"

# A matrix with a negative dominant eigenvalue (largest in magnitude, not value)
C = np.array([[-5.0, 0.0], [0.0, 1.0]])
lam_c, v_c = power_iteration(C, iters=50)
assert abs(abs(lam_c) - 5) < 1e-6, "power iteration finds the eigenvalue largest in magnitude"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def power_iteration(A, iters=100):
    A = np.asarray(A, dtype=float)
    v = np.ones(A.shape[0])
    for _ in range(iters):
        v = A @ v
        v = v / np.linalg.norm(v)
    lam = v @ A @ v
    return lam, v
```

</details>

---
## 🎯 Extra practice — matrix algebra toolkit

Four more from-scratch routines from
[DML-OpenProblem](https://github.com/Open-Deep-ML/DML-OpenProblem) that build directly
on eigendecomposition:

- **DML 6** — `calculate-eigenvalues-of-a-matrix` (sorted highest to lowest)
- **DML 8** — `calculate-2x2-matrix-inverse`
- **DML 13** — `determinant-of-a-4x4-matrix-using-laplace-s-expansion` (recursive
  cofactor expansion — no `np.linalg.det`)
- **DML 28** — `svd-of-a-2x2-matrix-using-eigen-values-vectors`

In [ ]:
def calculate_eigenvalues(matrix):
    """DML 6 — eigenvalues of a 2x2 matrix, sorted highest to lowest."""
    (a, b), (c, d) = matrix
    tr, det = a + d, a * d - b * c
    # TODO(you): same quadratic as eigvals_2x2 above — sqrt(tr^2 - 4*det)
    disc = np.sqrt(tr ** 2 - 4 * det)
    return [(tr + disc) / 2, (tr - disc) / 2]


def inverse_2x2(matrix):
    """DML 8 — inverse of a 2x2 matrix, or None if singular."""
    (a, b), (c, d) = matrix
    det = a * d - b * c
    # TODO(you): if det == 0 there is no inverse
    if det == 0:
        return None
    return [[d / det, -b / det], [-c / det, a / det]]


def _minor(matrix, i, j):
    """Delete row i and column j."""
    return [row[:j] + row[j + 1:] for k, row in enumerate(matrix) if k != i]


def determinant_recursive(matrix):
    """Laplace / cofactor expansion along the first row, any size."""
    n = len(matrix)
    if n == 1:
        return matrix[0][0]
    if n == 2:
        return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]
    det = 0
    for j in range(n):
        sign = (-1) ** j
        # TODO(you): add sign * matrix[0][j] * determinant of the minor with row 0, col j removed
        det += sign * matrix[0][j] * determinant_recursive(_minor(matrix, 0, j))
    return det


def determinant_4x4(matrix):
    """DML 13 — determinant of a 4x4 matrix via recursive Laplace expansion."""
    return determinant_recursive(matrix)


def svd_2x2(A):
    """DML 28 — SVD of a 2x2 matrix from the eigendecomposition of A^T A and A A^T."""
    A = np.asarray(A, dtype=float)
    AtA = A.T @ A
    # TODO(you): eigh gives eigenvalues ascending; sort descending for U,S,V convention
    eigvals, V = np.linalg.eigh(AtA)
    order = np.argsort(eigvals)[::-1]
    eigvals, V = eigvals[order], V[:, order]
    singular_values = np.sqrt(np.clip(eigvals, 0, None))
    U = np.zeros_like(A)
    for i in range(len(singular_values)):
        if singular_values[i] > 1e-12:
            # TODO(you): the i-th column of U is (A @ v_i) / sigma_i
            U[:, i] = (A @ V[:, i]) / singular_values[i]
    return U, singular_values, V.T

In [ ]:
# Checks — run me
assert calculate_eigenvalues([[2, 1], [1, 2]]) == [3.0, 1.0], "DML 6 example"
assert calculate_eigenvalues([[4, 1], [2, 3]]) == [5.0, 2.0], "sorted highest to lowest"

assert inverse_2x2([[4, 7], [2, 6]]) == [[0.6, -0.7], [-0.2, 0.4]], "DML 8 example"
assert inverse_2x2([[1, 2], [2, 4]]) is None, "a singular (rank-deficient) matrix has no inverse"

assert determinant_4x4([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]) == 0, "DML 13 example"
assert determinant_4x4(np.eye(4).tolist()) == 1, "the identity has determinant 1"
diag4 = [[2, 0, 0, 0], [0, 3, 0, 0], [0, 0, 4, 0], [0, 0, 0, 5]]
assert determinant_4x4(diag4) == 120, "a diagonal matrix's determinant is the product of its diagonal"

U, S, VT = svd_2x2([[-10, 8], [10, -1]])
assert np.allclose(U @ np.diag(S) @ VT, [[-10, 8], [10, -1]]), "DML 28 example: reconstruction must equal A"
assert np.allclose(U.T @ U, np.eye(2)), "U must be orthogonal"
assert np.allclose(VT @ VT.T, np.eye(2)), "V must be orthogonal"
assert np.allclose(sorted(S), [4.47213595, 15.65247584], atol=1e-6), "singular values match DML 28"

# Degenerate case: the zero matrix has all-zero singular values
U0, S0, VT0 = svd_2x2([[0, 0], [0, 0]])
assert np.allclose(S0, [0, 0]), "the zero matrix has zero singular values"

print("✅ Extra-practice matrix algebra toolkit passed")

<details>
<summary>💡 Show solution</summary>

```python
def calculate_eigenvalues(matrix):
    (a, b), (c, d) = matrix
    tr, det = a + d, a * d - b * c
    disc = np.sqrt(tr ** 2 - 4 * det)
    return [(tr + disc) / 2, (tr - disc) / 2]


def inverse_2x2(matrix):
    (a, b), (c, d) = matrix
    det = a * d - b * c
    if det == 0:
        return None
    return [[d / det, -b / det], [-c / det, a / det]]


def _minor(matrix, i, j):
    return [row[:j] + row[j + 1:] for k, row in enumerate(matrix) if k != i]


def determinant_recursive(matrix):
    n = len(matrix)
    if n == 1:
        return matrix[0][0]
    if n == 2:
        return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]
    det = 0
    for j in range(n):
        sign = (-1) ** j
        det += sign * matrix[0][j] * determinant_recursive(_minor(matrix, 0, j))
    return det


def determinant_4x4(matrix):
    return determinant_recursive(matrix)


def svd_2x2(A):
    A = np.asarray(A, dtype=float)
    AtA = A.T @ A
    eigvals, V = np.linalg.eigh(AtA)
    order = np.argsort(eigvals)[::-1]
    eigvals, V = eigvals[order], V[:, order]
    singular_values = np.sqrt(np.clip(eigvals, 0, None))
    U = np.zeros_like(A)
    for i in range(len(singular_values)):
        if singular_values[i] > 1e-12:
            U[:, i] = (A @ V[:, i]) / singular_values[i]
    return U, singular_values, V.T
```

</details>